# Deep Research with LangGraph

This notebook replicates the full deep research pipeline (`feedback.py` → `research.py` → `reporting.py`) using **LangGraph**.

## Graph Structure
- **State**: `DeepResearchState` — shared TypedDict passed between all nodes
- **Nodes (10)**: `collect_query` → `generate_feedback` → `collect_answers` → `combine_query` → `collect_params` → `initialize_research` → `process_research_batch` → `finalize_research` → `generate_report` → `save_report`
- **Edges (9 straight)**: sequential connections between nodes
- **Conditional Edge (1)**: after `process_research_batch`, loops back if queue is non-empty, else proceeds to `finalize_research`

## Cell 1 — Install Dependencies

In [ ]:
%pip install langgraph langchain langchain-openai firecrawl-py python-dotenv pydantic

## Cell 2 — Imports

In [1]:
import os
from datetime import datetime
from typing import List, Dict, Optional
from typing_extensions import TypedDict

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from pydantic import BaseModel
from firecrawl import FirecrawlApp

from langgraph.graph import StateGraph, START, END
from IPython.display import display, Image

## Cell 3 — Configuration

In [2]:
load_dotenv()

# gpt-4o-mini, gpt-4o, o3-mini 로 변경 가능 (structured output 지원 모델만 가능 → o1-mini는 불가)
FEEDBACK_MODEL = "gpt-4o-mini"
RESEARCH_MODEL = "gpt-4o"

# o3-mini, gpt-4o, gpt-4o-mini 로 변경 가능
REPORTING_MODEL = "gpt-4o"

print("설정 완료:", FEEDBACK_MODEL, "/", RESEARCH_MODEL, "/", REPORTING_MODEL)

## Cell 4 — Pydantic Models

In [3]:
# ── Step 1: Feedback ──────────────────────────────────────────────────────────
class FeedbackResponse(BaseModel):
    questions: List[str]


# ── Step 2: Research ──────────────────────────────────────────────────────────
class SearchResult(BaseModel):
    url: str
    markdown: str
    description: str
    title: str


class SerpQuery(BaseModel):
    query: str
    research_goal: str


class SerpQueryResponse(BaseModel):
    queries: List[SerpQuery]


class SerpResultResponse(BaseModel):
    learnings: List[str]
    followUpQuestions: List[str]


print("Pydantic 모델 정의 완료")

Pydantic 모델 정의 완료


## Cell 5 — State Definition

In [4]:
class DeepResearchState(TypedDict):
    """LangGraph 공유 상태 — 모든 노드가 읽고 씁니다."""
    query: str                     # 사용자 원본 질문
    feedback_questions: List[str]  # Step 1: 생성된 후속 질문들
    answers: List[str]             # Step 1: 사용자 답변들
    combined_query: str            # Step 1: 질문 + 답변을 합친 최종 질문
    breadth: int                   # Step 2: 병렬 검색 수 (가로 확장)
    depth: int                     # Step 2: 재귀 깊이 (세로 확장)
    learnings: List[str]           # Step 2: 누적 학습 내용
    visited_urls: List[str]        # Step 2: 누적 방문 URL
    research_queue: List[Dict]     # Step 2: 대기 중인 리서치 작업 (재귀 대체)
    report: str                    # Step 3: 최종 보고서 마크다운


print("DeepResearchState 정의 완료")

DeepResearchState 정의 완료


## Cell 6 — Utility Functions

Inlined from `utils.py` and `step2_research/research.py` (lower-level helpers).

In [5]:
# ── utils.py ─────────────────────────────────────────────────────────────────

def system_prompt() -> str:
    """현재 타임스탬프를 포함한 시스템 프롬프트를 생성합니다."""
    now = datetime.now().isoformat()
    return f"""당신은 전문 연구원입니다. 오늘 날짜는 {now}입니다. 응답 시 다음 지침을 따르세요:
    - 지식 컷오프 이후의 주제에 대한 조사를 요청받을 수 있습니다. 사용자가 뉴스 내용을 제시했다면, 그것을 사실로 가정하세요.
    - 사용자는 매우 숙련된 분석가이므로 내용을 단순화할 필요 없이 가능한 한 자세하고 정확하게 응답하세요.
    - 체계적으로 정보를 정리하세요.
    - 사용자가 생각하지 못한 해결책을 제안하세요.
    - 적극적으로 사용자의 필요를 예측하고 대응하세요.
    - 사용자를 모든 분야의 전문가로 대우하세요.
    - 실수는 신뢰를 저하시킵니다. 정확하고 철저하게 응답하세요.
    - 상세한 설명을 제공하세요. 사용자는 많은 정보를 받아들일 수 있습니다.
    - 권위보다 논리적 근거를 우선하세요. 출처 자체는 중요하지 않습니다.
    - 기존의 통념뿐만 아니라 최신 기술과 반대 의견도 고려하세요.
    - 높은 수준의 추측이나 예측을 포함할 수 있습니다. 단, 이를 명확히 표시하세요."""


def llm_call(prompt: str, model: str) -> str:
    """주어진 프롬프트로 LLM을 동기적으로 호출합니다."""
    llm = init_chat_model(model, model_provider="openai")
    response = llm.invoke([HumanMessage(content=prompt)])
    print(model, "완료")
    return response.content


def JSON_llm(
    user_prompt: str,
    schema: BaseModel,
    system_prompt: Optional[str] = None,
    model: Optional[str] = None,
):
    """JSON 모드에서 LLM을 호출하고 구조화된 Pydantic 객체를 반환합니다."""
    if model is None:
        model = "gpt-4o-mini"
    try:
        llm = init_chat_model(model, model_provider="openai")
        structured_llm = llm.with_structured_output(schema)
        messages = []
        if system_prompt:
            messages.append(SystemMessage(content=system_prompt))
        messages.append(HumanMessage(content=user_prompt))
        return structured_llm.invoke(messages)
    except Exception as e:
        print(f"Error in JSON_llm: {e}")
        return None


# ── research.py helpers ───────────────────────────────────────────────────────

def firecrawl_search(query: str, timeout: int = 15000, limit: int = 5) -> List[SearchResult]:
    """Firecrawl 검색 API를 호출하여 결과를 반환합니다."""
    try:
        app = FirecrawlApp(api_key=os.getenv("FIRECRAWL_API_KEY", ""))
        response = app.search(
            query=query,
            timeout=timeout,
            limit=limit,
            scrape_options={"formats": ["markdown"]},
        )
        results = []
        for item in (response.web or []):
            if hasattr(item, "metadata") and item.metadata:
                results.append(SearchResult(
                    url=item.metadata.url or "",
                    markdown=item.markdown or "",
                    description=item.metadata.description or "",
                    title=item.metadata.title or "",
                ))
            else:
                results.append(SearchResult(
                    url=getattr(item, "url", "") or "",
                    markdown=getattr(item, "markdown", "") or "",
                    description=getattr(item, "description", "") or "",
                    title=getattr(item, "title", "") or "",
                ))
        return results
    except Exception as e:
        print(f"Firecrawl 검색 오류: {e}")
        return []


def generate_serp_queries(
    query: str,
    num_queries: int = 3,
    learnings: Optional[List[str]] = None,
) -> List[SerpQuery]:
    """사용자 쿼리와 이전 학습 내용을 바탕으로 SERP 검색 쿼리를 생성합니다."""
    prompt = (
        f"다음 사용자 입력을 기반으로 연구 주제를 조사하기 위한 SERP 검색 쿼리를 생성하세요. "
        f"JSON 객체를 반환하며, 'queries' 배열 필드에 {num_queries}개의 검색 쿼리를 포함해야 합니다 (쿼리가 명확할 경우 더 적을 수도 있음). "
        f"각 쿼리 객체에는 'query'와 'research_goal' 필드가 포함되어야 하며, 각 쿼리는 고유해야 합니다: "
        f"<입력>{query}</입력>"
    )
    if learnings:
        prompt += f"\n\n다음은 이전 연구에서 얻은 학습 내용입니다. 이를 활용하여 더 구체적인 쿼리를 생성하세요: {' '.join(learnings)}"

    sys_prompt = system_prompt()
    response_json = JSON_llm(prompt, SerpQueryResponse, system_prompt=sys_prompt, model=RESEARCH_MODEL)
    try:
        result = SerpQueryResponse.model_validate(response_json)
        queries = result.queries if result.queries else []
        print(f"리서치 주제에 대한 SERP 검색 쿼리 {len(queries)}개 생성됨")
        return queries[:num_queries]
    except Exception as e:
        print(f"오류: generate_serp_queries에서 JSON 응답을 처리하는 중 오류 발생: {e}")
        print(f"원시 응답: {response_json}")
        return []


def process_serp_result(
    query: str,
    search_result: List[SearchResult],
    num_learnings: int = 5,
    num_follow_up_questions: int = 3,
) -> Dict[str, List[str]]:
    """검색 결과를 처리하여 학습 내용과 후속 질문을 추출합니다."""
    contents = [
        item.markdown.strip()[:25000]
        for item in search_result if item.markdown
    ]
    contents_str = "".join(f"<내용>\n{content}\n</내용>" for content in contents)
    prompt = (
        f"다음은 쿼리 <쿼리>{query}</쿼리>에 대한 SERP 검색 결과입니다. "
        f"이 내용을 바탕으로 학습 내용을 추출하고 후속 질문을 생성하세요. "
        f"JSON 객체로 반환하며, 'learnings' 및 'followUpQuestions' 키를 포함한 배열을 반환하세요. "
        f"각 학습 내용은 고유하고 간결하며 정보가 풍부해야 합니다. 최대 {num_learnings}개의 학습 내용과 "
        f"{num_follow_up_questions}개의 후속 질문을 포함해야 합니다.\n\n"
        f"<검색 결과>{contents_str}</검색 결과>"
    )
    sys_prompt = system_prompt()
    response_json = JSON_llm(prompt, SerpResultResponse, system_prompt=sys_prompt, model=RESEARCH_MODEL)
    try:
        result = SerpResultResponse.model_validate(response_json)
        return {
            "learnings": result.learnings,
            "followUpQuestions": result.followUpQuestions[:num_follow_up_questions],
        }
    except Exception as e:
        print(f"오류: process_serp_result에서 JSON 응답을 처리하는 중 오류 발생: {e}")
        print(f"원시 응답: {response_json}")
        return {"learnings": [], "followUpQuestions": []}


print("유틸리티 함수 정의 완료")

## Cell 7 — Step 1 Nodes

Replicates `step1_feedback/feedback.py` and the user-interaction logic in `main.py`.

In [6]:
# ── Node 1: collect_query ─────────────────────────────────────────────────────
def collect_query(state: DeepResearchState) -> dict:
    """사용자로부터 초기 연구 질문을 입력받습니다."""
    query = input("어떤 주제에 대해 리서치하시겠습니까?: ")
    return {"query": query}


# ── Node 2: generate_feedback ─────────────────────────────────────────────────
def generate_feedback(state: DeepResearchState) -> dict:
    """연구 방향을 명확히 하기 위한 후속 질문을 생성합니다. (feedback.py)"""
    print("------------------------------------------1단계: 추가 질문 생성----------------------------------------------------")
    query = state["query"]
    max_feedbacks = 3

    prompt = f"""
    Given the following query from the user, ask some follow up questions to clarify the research direction. Return a maximum of ${max_feedbacks} questions, but feel free to return less if the original query is clear.
    ask the follow up questions in korean
    <query>${query}</query>`
    """

    response = JSON_llm(prompt, FeedbackResponse, system_prompt=system_prompt(), model=FEEDBACK_MODEL)

    try:
        if response is None:
            print("오류: JSON_llm이 None을 반환했습니다.")
            return {"feedback_questions": []}
        questions = response.questions
        print(f"주제 '{query}'에 대한 후속 질문 {len(questions)}개 생성됨")
        print(f"생성된 후속 질문: {questions}")
        return {"feedback_questions": questions}
    except Exception as e:
        print(f"오류: JSON 응답 처리 중 문제 발생: {e}")
        print(f"원시 응답: {response}")
        return {"feedback_questions": []}


# ── Node 3: collect_answers ───────────────────────────────────────────────────
def collect_answers(state: DeepResearchState) -> dict:
    """생성된 후속 질문에 대한 사용자 답변을 수집합니다."""
    feedback_questions = state["feedback_questions"]
    answers = []

    if feedback_questions:
        print("\n다음 질문에 답변해 주세요:")
        for idx, question in enumerate(feedback_questions, start=1):
            answer = input(f"질문 {idx}: {question}\n답변: ")
            answers.append(answer)
    else:
        print("추가 질문이 생성되지 않았습니다.")

    return {"answers": answers}


# ── Node 4: combine_query ─────────────────────────────────────────────────────
def combine_query(state: DeepResearchState) -> dict:
    """초기 질문과 후속 질문/답변을 하나의 combined_query 문자열로 합칩니다."""
    query = state["query"]
    feedback_questions = state["feedback_questions"]
    answers = state["answers"]

    combined = f"초기 질문: {query}\n"
    for i in range(len(feedback_questions)):
        combined += f"\n{i+1}. 질문: {feedback_questions[i]}\n"
        combined += f"   답변: {answers[i]}\n"

    print("최종질문 : \n")
    print(combined)
    return {"combined_query": combined}


# ── Node 5: collect_params ────────────────────────────────────────────────────
def collect_params(state: DeepResearchState) -> dict:
    """연구 범위(breadth)와 깊이(depth)를 사용자로부터 입력받습니다."""
    try:
        breadth = int(input("연구 범위를 입력하세요 (예: 2): ") or "2")
    except ValueError:
        breadth = 2
    try:
        depth = int(input("연구 깊이를 입력하세요 (예: 2): ") or "2")
    except ValueError:
        depth = 2
    return {"breadth": breadth, "depth": depth}


print("Step 1 노드 정의 완료")

## Cell 8 — Step 2 Nodes

Replicates `step2_research/research.py` (`deep_research` recursion → queue + conditional edge).

**Queue design**: each item `{query, breadth, depth}` corresponds to one recursive call of the original `deep_research()`. The conditional edge loops `process_research_batch` until the queue is empty.

In [7]:
# ── Node 6: initialize_research ───────────────────────────────────────────────
def initialize_research(state: DeepResearchState) -> dict:
    """초기 리서치 태스크를 큐에 등록하고 누적 상태를 초기화합니다."""
    print("------------------------------------------2단계: 딥리서치----------------------------------------------------")
    initial_task = {
        "query": state["combined_query"],
        "breadth": state["breadth"],
        "depth": state["depth"],
    }
    return {
        "research_queue": [initial_task],
        "learnings": [],
        "visited_urls": [],
    }


# ── Node 7: process_research_batch ────────────────────────────────────────────
def process_research_batch(state: DeepResearchState) -> dict:
    """
    큐에서 태스크 하나를 꺼내 처리합니다 — 원본 deep_research()의 재귀 한 단계에 해당.

    1. Pop task from research_queue
    2. Generate SERP queries (breadth 개)
    3. For each SERP query:
       a. firecrawl_search → search results
       b. process_serp_result → learnings + follow-up questions
       c. If depth > 0: push follow-up task onto queue
    4. Return updated queue, learnings, visited_urls
    """
    queue = list(state["research_queue"])
    task = queue.pop(0)  # FIFO — 원본의 재귀 순서와 동일

    query   = task["query"]
    breadth = task["breadth"]
    depth   = task["depth"]

    # 누적 learnings / urls (원본 재귀에서 전달하는 변수와 동일)
    level_learnings = list(state["learnings"])
    level_urls      = list(state["visited_urls"])

    print(f" ---------- Deep Research 시도 ------------------")
    print(f" <주제> \n {query} \n </주제>")

    serp_queries = generate_serp_queries(
        query=query,
        num_queries=breadth,
        learnings=level_learnings,
    )
    print(f" ------------ 해당 <주제>에 대해서 생성된 검색 키워드 ({len(serp_queries)}개 생성)------------")
    print(f" {serp_queries} \n")

    for index, serp_query in enumerate(serp_queries, start=1):
        result = firecrawl_search(serp_query.query)
        new_urls = [item.url for item in result if item.url]

        serp_result = process_serp_result(
            query=serp_query.query,
            search_result=result,
            num_follow_up_questions=breadth,  # 원본과 동일: breadth 사용
        )

        print(f"  - 의 {index}번째 검색 키워드 ({serp_query.query})에 대한 조사 완료")
        print(f"  - 조사완료된 URL들:")
        for url in new_urls:
            print(f"    • {url}")
        print()
        print(f"  - 조사로 얻은 학습 내용 ({len(serp_result['learnings'])}개 생성) : \n {serp_result['learnings']} \n")

        all_learnings = level_learnings + serp_result["learnings"]
        all_urls      = level_urls      + new_urls
        new_depth     = depth - 1
        new_breadth   = max(1, breadth // 2)

        if new_depth > 0:
            # 원본 재귀 호출 → 큐에 추가 (conditional edge가 루프를 담당)
            next_query = (
                f"이전 연구목표: {serp_query.research_goal}\n"
                f"후속 연구방향: {' '.join(serp_result['followUpQuestions'])}"
            )
            queue.append({"query": next_query, "breadth": new_breadth, "depth": new_depth})

        level_learnings = all_learnings
        level_urls      = all_urls

    return {
        "research_queue": queue,
        "learnings": level_learnings,
        "visited_urls": level_urls,
    }


# ── Conditional Edge: should_continue_research ────────────────────────────────
def should_continue_research(state: DeepResearchState) -> str:
    """
    큐에 남은 태스크가 있으면 process_research_batch로 루프,
    없으면 finalize_research로 진행합니다.
    """
    if state["research_queue"]:
        return "process_research_batch"  # 루프 계속
    return "finalize_research"           # 리서치 완료


# ── Node 8: finalize_research ─────────────────────────────────────────────────
def finalize_research(state: DeepResearchState) -> dict:
    """학습 내용과 방문 URL을 중복 제거합니다 (원본 deep_research 마지막 단계)."""
    deduped_learnings = list(set(state["learnings"]))
    deduped_urls      = list(set(state["visited_urls"]))

    print("\n연구 결과:")
    for learning in deduped_learnings:
        print(f" - {learning}")

    return {
        "learnings": deduped_learnings,
        "visited_urls": deduped_urls,
    }


print("Step 2 노드 정의 완료")

Step 2 노드 정의 완료


## Cell 9 — Step 3 Nodes

Replicates `step3_reporting/reporting.py` and the file-save logic in `main.py`.

In [8]:
# ── Node 9: generate_report ───────────────────────────────────────────────────
def generate_report(state: DeepResearchState) -> dict:
    """모든 연구 결과를 바탕으로 최종 마크다운 보고서를 생성합니다. (reporting.py)"""
    print("------------------------------------------3단계: 보고서 작성----------------------------------------------------")

    learnings     = state["learnings"]
    visited_urls  = state["visited_urls"]
    prompt        = state["combined_query"]

    learnings_string = (
        "\n".join([f"<learning>\n{learning}\n</learning>" for learning in learnings])
    ).strip()[:150000]

    user_prompt = (
        f"사용자가 제시한 다음 프롬프트에 대해, 러서치 결과를 바탕으로 최종 보고서를 작성하세요. "
        f"마크다운 형식으로 상세한 보고서(6,000자 이상)를 작성하세요. "
        f"러서치에서 얻은 모든 학습 내용을 포함해야 합니다:\n\n"
        f"<prompt>{prompt}</prompt>\n\n"
        f"다음은 리서치를 통해 얻은 모든 학습 내용입니다:\n\n<learnings>\n{learnings_string}\n</learnings>"
    )

    sys_prompt = system_prompt()
    if sys_prompt:
        user_prompt = f"{sys_prompt}\n\n{user_prompt}"

    try:
        report = llm_call(user_prompt, REPORTING_MODEL)
        urls_section = "\n\n## 출처\n\n" + "\n".join(f"- {url}" for url in visited_urls)
        full_report = report + urls_section
    except Exception as e:
        print(f"Error generating report: {e}")
        full_report = "Error generating report"

    return {"report": full_report}


# ── Node 10: save_report ──────────────────────────────────────────────────────
def save_report(state: DeepResearchState) -> dict:
    """최종 보고서를 출력하고 output/output.md 에 저장합니다."""
    report = state["report"]

    print("\n최종 보고서:\n")
    print(report)

    os.makedirs("output", exist_ok=True)
    with open("output/output.md", "w", encoding="utf-8") as f:
        f.write(report)

    print("\n보고서가 output/output.md 파일에 저장되었습니다.")
    return {}


print("Step 3 노드 정의 완료")

## Cell 10 — Graph Construction

Assembles the LangGraph with:
- 10 nodes
- 9 straight edges
- 1 conditional edge (research loop)

In [9]:
# ── Build the graph ───────────────────────────────────────────────────────────
builder = StateGraph(DeepResearchState)

# Add nodes
builder.add_node("collect_query",           collect_query)
builder.add_node("generate_feedback",       generate_feedback)
builder.add_node("collect_answers",         collect_answers)
builder.add_node("combine_query",           combine_query)
builder.add_node("collect_params",          collect_params)
builder.add_node("initialize_research",     initialize_research)
builder.add_node("process_research_batch",  process_research_batch)
builder.add_node("finalize_research",       finalize_research)
builder.add_node("generate_report",         generate_report)
builder.add_node("save_report",             save_report)

# Straight edges (9)
builder.add_edge(START,                    "collect_query")
builder.add_edge("collect_query",          "generate_feedback")
builder.add_edge("generate_feedback",      "collect_answers")
builder.add_edge("collect_answers",        "combine_query")
builder.add_edge("combine_query",          "collect_params")
builder.add_edge("collect_params",         "initialize_research")
builder.add_edge("initialize_research",    "process_research_batch")
builder.add_edge("finalize_research",      "generate_report")
builder.add_edge("generate_report",        "save_report")
builder.add_edge("save_report",            END)

# Conditional edge (1) — research loop
builder.add_conditional_edges(
    "process_research_batch",
    should_continue_research,
    {
        "process_research_batch": "process_research_batch",  # 루프
        "finalize_research":      "finalize_research",        # 완료
    },
)

# Compile
graph = builder.compile()

print("그래프 빌드 완료")
print(f"  노드: {list(graph.nodes.keys())}")

## Cell 11 — Graph Visualization

In [10]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    # pygraphviz / graphviz 미설치 시 mermaid 텍스트 출력
    print("그래프 PNG 렌더링 실패 (graphviz 필요):", e)
    print()
    print(graph.get_graph().draw_mermaid())

## Cell 12 — Run

> **참고**: Firecrawl 무료 티어는 분당 5회 요청 제한이 있습니다.  
> `breadth=2, depth=2` 설정을 권장합니다.

In [11]:
final_state = graph.invoke({})
print("\n파이프라인 완료!")